In [29]:
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans


suffix = 'covariate_high_resource'


X_test = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/X_test_%s.joblib'%suffix)
y_test = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/orig_y_test_%s.joblib'%suffix)
binary_y_test = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/binary_y_test_%s.joblib'%suffix)

X_train = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/X_train_%s.joblib'%suffix)
y_train = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/y_train_%s.joblib'%suffix)
orig_y_train = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/orig_y_train_%s.joblib'%suffix)
binary_y_train = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/binary_y_train_%s.joblib'%suffix)

X_val = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/X_val_%s.joblib'%suffix)
y_val = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/y_val_%s.joblib'%suffix)
orig_y_val = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/orig_y_val_%s.joblib'%suffix)
binary_y_val = joblib.load('$CWITE_DATA_ROOT/support50_propbin_data/binary_y_val_%s.joblib'%suffix)

k_values = range(2, 50)
inertia_scores = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_train)
    inertia_scores.append(kmeans.inertia_)

inertia_diff = np.diff(inertia_scores)
inertia_diff2 = np.diff(inertia_diff)
elbow_k = k_values[np.argmin(inertia_diff2) + 1]  # +1 to index properly

# --- Step 2: Fit with elbow_k and apply to test ---
kmeans = KMeans(n_clusters=elbow_k, random_state=42, n_init=10)
categories_train = kmeans.fit_predict(X_train)
categories_val = kmeans.predict(X_val)

# --- Step 3: Cluster-wise analysis on test ---


splits = {
    '<20': orig_y_val < 20,
    '≥20': orig_y_val >= 20
}

for split_name, mask in splits.items():
    diff_var = []
    min_gaps = []

    for cluster_id in range(elbow_k):
        cluster_mask = (categories_val == cluster_id) & mask
        if not np.any(cluster_mask):
            continue

        y_cluster = y_val[cluster_mask]
        orig_y_cluster = orig_y_val[cluster_mask]
        binary_cluster = binary_y_val[cluster_mask]

        cens_times = y_cluster[binary_cluster == 0]
        uncens_times = y_cluster[binary_cluster == 1]
        orig_cens_times = orig_y_cluster[binary_cluster == 0]

        if len(cens_times) > 0:
            diff_var.append(np.var(cens_times) - np.var(orig_cens_times))
            min_gaps.append(np.min(np.abs(cens_times - orig_cens_times)))

    print(f"\nSplit: {split_name}")
    print(f"Avg ΔVar (censored - uncensored): {np.min(diff_var):.2f}")
    print(f"Avg min gap (cens vs. orig): {np.max(min_gaps):.2f}")



Split: <20
Avg ΔVar (censored - uncensored): -25.34
Avg min gap (cens vs. orig): 0.00

Split: ≥20
Avg ΔVar (censored - uncensored): -30.44
Avg min gap (cens vs. orig): 6.00


In [24]:
diff_var

[27.273466198979584,
 41.6572705804202,
 17.723257023933385,
 50.18250000000001,
 102.0495867768595,
 90.11176857330705,
 63.83623609885741]